# imports

In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from pathlib import Path
import json
import os
from tqdm import tqdm
import pandas as pd

import snap.models as models
from snap.nsd_data import load_pandas

# load model + categories

In [1]:
model_name = 'resnet50'
trained = True
device = 'cuda'

In [11]:
model_kwargs = {'name': model_name,
                'pretrained': trained,
                'device': device}
model, layers, identifier, img_transforms = models.get_model(**model_kwargs)
model.eval()
model.cuda()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [13]:
imagenet_classes = models.ResNet50_Weights.DEFAULT.meta["categories"]

# load images

In [4]:
voxel_set_name = 'EVC-OTC'
voxel_set = ['EVC','OTC']
image_set = 'shared1000'
path_dir = '/mnt/ceph/users/alargen/small_nsd/DeepJuiceDev/juicyfruits/nsd_subset'

stimulus_path = f'{path_dir}/stimulus/{image_set}.csv'
image_root = os.path.join(path_dir, 'stimulus', image_set)

In [9]:
stimulus_data = load_pandas(stimulus_path)
stimulus_data['image_path'] = image_root + '/' + stimulus_data.image_name

In [10]:
stimulus_data.columns

Index(['image_id', 'image_name', 'coco_supercategs', 'coco_categs',
       'coco_areas', 'coco_captions', 'image_path'],
      dtype='object')

# check top 5 accuracy

In [16]:
top5_correct = 0
total = 0

for _, row in tqdm(stimulus_data.iterrows(), total=len(stimulus_data)):
    image_path = row['image_path']
    true_labels = row['coco_categs']  # List of categories

    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Skipping {image_path}: {e}")
        continue

    input_tensor = img_transforms(image).unsqueeze(0).cuda()

    with torch.no_grad():
        output = model(input_tensor)
        top5_probs, top5_indices = torch.topk(output, 5)
        top5_labels = [imagenet_classes[idx] for idx in top5_indices[0]]

    # Check if any of the top-5 predictions are in the ground-truth categories
    if any(label in true_labels for label in top5_labels):
        top5_correct += 1
    total += 1
    print(top5_labels, true_labels)
    raise Exception

top5_accuracy = top5_correct / total
print(f"Top-5 Accuracy: {top5_accuracy:.3f}")

  0%|          | 0/1000 [00:00<?, ?it/s]


['plate', 'hot pot', 'hen-of-the-woods', 'butternut squash', 'spaghetti squash'] ['spoon', 'broccoli', 'broccoli', 'carrot', 'carrot', 'carrot', 'carrot', 'carrot', 'carrot', 'bowl', 'carrot', 'carrot', 'carrot', 'carrot']


Exception: 

In [21]:
stimulus_data['coco_categs'][0]

"['spoon', 'broccoli', 'broccoli', 'carrot', 'carrot', 'carrot', 'carrot', 'carrot', 'carrot', 'bowl', 'carrot', 'carrot', 'carrot', 'carrot']"